## Imports + warnings

In [58]:
import numpy as np
import pandas as pd
import joblib
import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)


## Load saved models

In [59]:
REGION_MODEL_PATH = "region_model.pkl"
UPPER_EXPERT_PATH = "upper_expert.pkl"
LOWER_EXPERT_PATH = "lower_expert.pkl"
OTHER_EXPERT_PATH = "other_expert.pkl"

region_bundle = joblib.load(REGION_MODEL_PATH)
upper_bundle  = joblib.load(UPPER_EXPERT_PATH)
lower_bundle  = joblib.load(LOWER_EXPERT_PATH)
other_bundle  = joblib.load(OTHER_EXPERT_PATH)

region_model = region_bundle["model"]
feature_cols = region_bundle["feature_cols"]   # single source of truth

upper_model = upper_bundle["model"]
lower_model = lower_bundle["model"]
other_model = other_bundle["model"]

print("Loaded models.")
print("n_features:", len(feature_cols))
print("Upper classes:", upper_bundle["classes"])
print("Lower classes:", lower_bundle["classes"])
print("Other classes:", other_bundle["classes"])


Loaded models.
n_features: 38
Upper classes: ['ForwardHead', 'LeftArmFallForward', 'LeftShoulderElevation', 'RightArmFallForward', 'RightShoulderElevation']
Lower classes: ['LeftKneeMovesOutward', 'Lower__RARE', 'RightKneeMovesInward', 'RightKneeMovesOutward']
Other classes: ['ExcessiveForwardLean', 'LeftAsymmetricalWeightShift', 'RightAsymmetricalWeightShift']


## align/renormalize probabilities into full-class space

In [60]:
def _safe_as_list(x):
    return list(x) if isinstance(x, (list, tuple, np.ndarray, pd.Index)) else [x]

def align_proba_to_all_classes(proba, model_classes, all_classes, renormalize=True, eps=1e-12):
    model_classes = _safe_as_list(model_classes)
    all_classes = _safe_as_list(all_classes)

    out = np.zeros((proba.shape[0], len(all_classes)), dtype=float)
    idx = {c: i for i, c in enumerate(all_classes)}

    for j, c in enumerate(model_classes):
        if c in idx:
            out[:, idx[c]] = proba[:, j]

    if renormalize:
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums = np.maximum(row_sums, eps)
        out = out / row_sums

    return out


## Core inference (hard routing OR soft routing)

In [61]:
def predict_weakest_link(
    X_input,
    routing="soft",   # "soft" or "hard"
    top_k=5
):
    """
    Accepts:
        - list of 38 feature values
        - numpy array shape (38,)
        - dict {feature: value}
        - pandas DataFrame

    Returns:
        final_prediction (str),
        top_k_list = [(class, prob), ...]
    """

    # -------------------------
    # Convert input to DataFrame
    # -------------------------
    if isinstance(X_input, (list, np.ndarray)):
        if len(X_input) != len(feature_cols):
            raise ValueError(f"Expected {len(feature_cols)} features, got {len(X_input)}")
        Xdf = pd.DataFrame([X_input], columns=feature_cols)

    elif isinstance(X_input, dict):
        Xdf = pd.DataFrame([X_input])[feature_cols]

    elif isinstance(X_input, pd.Series):
        Xdf = X_input.to_frame().T[feature_cols]

    elif isinstance(X_input, pd.DataFrame):
        Xdf = X_input[feature_cols].copy()

    else:
        raise ValueError("Unsupported input type.")

    Xdf = Xdf.apply(pd.to_numeric, errors="coerce")

    # -------------------------
    # Region prediction
    # -------------------------
    region_proba = region_model.predict_proba(Xdf)
    region_classes = list(region_model.named_steps["clf"].classes_)
    
    region_argmax = int(np.argmax(region_proba, axis=1)[0])
    region_pred = region_classes[region_argmax]


    # -------------------------
    # Expert predictions
    # -------------------------
    pu = upper_model.predict_proba(Xdf)
    pl = lower_model.predict_proba(Xdf)
    po = other_model.predict_proba(Xdf)

    upper_classes = upper_bundle["classes"]
    lower_classes = lower_bundle["classes"]
    other_classes = other_bundle["classes"]

    all_classes = sorted(set(upper_classes) | set(lower_classes) | set(other_classes))

    def align(proba, model_classes):
        out = np.zeros((proba.shape[0], len(all_classes)))
        idx = {c: i for i, c in enumerate(all_classes)}
        for j, c in enumerate(model_classes):
            if c in idx:
                out[:, idx[c]] = proba[:, j]
        return out

    pu_aligned = align(pu, upper_classes)
    pl_aligned = align(pl, lower_classes)
    po_aligned = align(po, other_classes)

    # -------------------------
    # Combine routing
    # -------------------------
    reg_idx = {c: i for i, c in enumerate(region_classes)}

    p_upper = region_proba[:, reg_idx["Upper"]] if "Upper" in reg_idx else np.zeros(region_proba.shape[0])
    p_lower = region_proba[:, reg_idx["Lower"]] if "Lower" in reg_idx else np.zeros(region_proba.shape[0])
    p_other = region_proba[:, reg_idx["Other"]] if "Other" in reg_idx else np.zeros(region_proba.shape[0])



    full_proba = np.zeros((len(Xdf), len(all_classes)))


    if routing == "soft":
        full_proba += p_upper[:, None] * pu_aligned
        full_proba += p_lower[:, None] * pl_aligned
        full_proba += p_other[:, None] * po_aligned

        s = full_proba.sum(axis=1, keepdims=True)
        s[s == 0] = 1.0
        full_proba = full_proba / s

    elif routing == "hard":
        reg = region_pred
        if reg == "Upper":
            full_proba = pu_aligned
        elif reg == "Lower":
            full_proba = pl_aligned
        else:
            full_proba = po_aligned

        s = full_proba.sum(axis=1, keepdims=True)
        s[s == 0] = 1.0
        full_proba = full_proba / s

    else:
        raise ValueError("routing must be 'soft' or 'hard'")

    # -------------------------
    # Final prediction
    # -------------------------
    probs = full_proba[0]
    final_pred = all_classes[np.argmax(probs)]

    # Top-k
    top_indices = np.argsort(-probs)[:top_k]
    top = [(all_classes[i], float(probs[i])) for i in top_indices]

    return final_pred, top


## Convenience: pretty output for one sample (top-k)

In [62]:
def show_prediction(result, row_idx=0, topk=5):
    region_pred = result["region_pred"][row_idx]
    final_pred  = result["final_pred"][row_idx]

    region_classes = result["region_classes"]
    region_proba = result["region_proba"][row_idx]

    print("Region prediction:", region_pred)
    print("Region probabilities:")
    for c, p in sorted(zip(region_classes, region_proba), key=lambda x: -x[1]):
        print(f"  {c:>5}: {p:.4f}")

    print("\nFinal weakest-link prediction:", final_pred)

    if "final_proba" in result:
        all_classes = result["all_classes"]
        probs = result["final_proba"][row_idx]
        top = sorted(zip(all_classes, probs), key=lambda x: -x[1])[:topk]
        print(f"\nTop-{topk} final classes:")
        for c, p in top:
            print(f"  {c:>28}: {p:.4f}")


## Example usage

### Helper cell

In [63]:
def parse_european_feature_string(s: str):
    """
    Converts a tab-separated string using comma decimals
    into a Python list of floats.
    """
    # Remove surrounding quotes and whitespace
    s = s.strip().strip('"').strip()
    
    # Split on whitespace (tabs or spaces)
    parts = s.split()
    
    # Replace decimal commas with dots and convert to float
    values = [float(p.replace(",", ".")) for p in parts]
    
    return values


### 4 examples

In [64]:
raw_1 = """0,364418938	0,269249163	0,665231946	0,599234816	0,69010043	0,565758011	0,462936394	0,521759923	0,899569584	0,989478718	0,635581062	0,81252989	0,762314682	0,69010043	0,462936394	0,364418938	0,989478718	0,762314682	0,735054998	0,736011478	0,511716882	0,245815399	0,286465806	0,864658058	0,618842659	0,544715447	0,978000956	0,728837877	0,853658537	0,772357724	0,441415591	0,865136298	0,656623625	0,642276423	0,552845528	0,648971784	0,949306552	0,308943089"""

raw_2 = """0,958871353	0,979913917	0,494978479	0,706838833	0,334289813	0,140124342	0,408417025	0,521759923	0,741272119	0,754184601	0,635581062	0,528933525	0,74796748	0,334289813	0,408417025	0,958871353	0,754184601	0,74796748	0,599713056	0,986131038	0,93830703	0,890961263	0,809182209	0,409373505	0,321377331	0,654710665	0,603538977	0,459110473	0,411286466	0,661884266	0,594452415	0,74988044	0,656623625	0,642276423	0,708273553	0,648971784	0,578192253	0,785270206"""

raw_3 = """0,858440937	0,901960784	0,995695839	0,993782879	0,205643233	0,93782879	0,68770923	0,521759923	0,156384505	0,471544715	0,93974175	0,798660928	0,178861789	0,205643233	0,68770923	0,858440937	0,471544715	0,178861789	0,9976088	0,164992826	0,613582018	0,663318986	0,286465806	0,710664754	0,737446198	0,619799139	0,092778575	0,459110473	0,284552846	0,99904352	0,441415591	0,806791009	0,728359637	0,985652798	0,552845528	0,648971784	0,578192253	0,670014347"""

raw_4 = """0,279770445	0,327594452	0,346724055	0,18890483	0,549497848	0,296508847	0,596365375	0,521759923	0,860353898	0,821616451	0,722620756	0,747011	0,890004782	0,549497848	0,596365375	0,279770445	0,821616451	0,890004782	0,471066475	0,37780966	0,512673362	0,707317073	0,684361549	0,870875179	0,980392157	0,544715447	0,730750837	0,581539933	0,854615017	0,420851267	0,649450024	0,670970827	0,838833094	0,642276423	0,692491631	0,817312291	0,922525108	0,308943089"""

raw_5 = """0,358680057	0,641798183	0,495456719	0,883309421	0,300813008	0,901960784	0,230989957	0,548541368	0,307986609	0,424198948	0,870875179	0,401721664	0,442850311	0,300813008	0,230989957	0,358680057	0,424198948	0,442850311	0,890004782	0,464371114	0,445241511	0,286465806	0,621712099	0,602582496	0,607364897	0,666188427	0,4988044	0,459110473	0,397895744	0,449067432	0,523194644	0,846006695	0,656623625	0,642276423	0,703969393	0,648971784	0,578192253	0,594930655"""

example_1 = parse_european_feature_string(raw_1)
example_2 = parse_european_feature_string(raw_2)
example_3 = parse_european_feature_string(raw_3)
example_4 = parse_european_feature_string(raw_4)
example_5 = parse_european_feature_string(raw_5)

print(len(example_1), len(example_2), len(example_3), len(example_4))


38 38 38 38


### Example 1 (Right answer: LeftArmFallForward)

In [65]:
pred, top = predict_weakest_link(example_1, top_k=5)

print("Predicted WeakestLink:", pred)
for cls, p in top:
    print(f"{cls:30s} {p:.3f}")


Predicted WeakestLink: LeftArmFallForward
LeftArmFallForward             0.819
RightShoulderElevation         0.180
RightKneeMovesOutward          0.001
ExcessiveForwardLean           0.000
LeftKneeMovesOutward           0.000


### Example 2 (Right answer: ForwardHead)

In [66]:
pred, top = predict_weakest_link(example_2, top_k=5)

print("Predicted WeakestLink:", pred)
for cls, p in top:
    print(f"{cls:30s} {p:.3f}")


Predicted WeakestLink: RightKneeMovesOutward
RightKneeMovesOutward          0.984
LeftAsymmetricalWeightShift    0.006
RightArmFallForward            0.006
LeftKneeMovesOutward           0.004
RightKneeMovesInward           0.000


### Example 3 (Right answer: Tie, both LeftAsymmetricalWeightShift and RightArmFallForward

In [67]:
pred, top = predict_weakest_link(example_3, top_k=5)

print("Predicted WeakestLink:", pred)
for cls, p in top:
    print(f"{cls:30s} {p:.3f}")


Predicted WeakestLink: RightArmFallForward
RightArmFallForward            0.761
RightKneeMovesOutward          0.235
LeftShoulderElevation          0.003
LeftAsymmetricalWeightShift    0.000
Lower__RARE                    0.000


### Example 4 (Right answer: LeftKneeMovesOutward)

In [68]:
pred, top = predict_weakest_link(example_4, top_k=5)

print("Predicted WeakestLink:", pred)
for cls, p in top:
    print(f"{cls:30s} {p:.3f}")


Predicted WeakestLink: LeftArmFallForward
LeftArmFallForward             1.000
LeftAsymmetricalWeightShift    0.000
RightArmFallForward            0.000
RightShoulderElevation         0.000
ExcessiveForwardLean           0.000


### Example 5 (Right answer: LeftShoulderElevation)

In [69]:
pred, top = predict_weakest_link(example_5, top_k=5)

print("Predicted WeakestLink:", pred)
for cls, p in top:
    print(f"{cls:30s} {p:.3f}")


Predicted WeakestLink: RightArmFallForward
RightArmFallForward            1.000
LeftArmFallForward             0.000
RightKneeMovesInward           0.000
LeftShoulderElevation          0.000
RightShoulderElevation         0.000


### Confusion matrix